# Module 6: Evals (13 min)

Run automated evaluations against the customer service agent — an **output eval** (is the response good?) and a **trajectory eval** (did the agent follow the right workflow?).

**Prerequisites:** Modules 1-5 completed

In [ ]:
!pip install -q -r requirements.txt

# The evals framework calls asyncio.run() internally, which clashes with the
# event loop Jupyter already runs. nest_asyncio patches the loop so the
# synchronous run_evaluations() API works inside a notebook cell.
import nest_asyncio

nest_asyncio.apply()

---

## Part 1: Output Evaluation — Is the Response Good?

The `OutputEvaluator` uses LLM-as-a-judge to score agent responses against expected outputs.

An eval is only useful if it can tell a **good** answer from a **bad** one. To prove that,
we run the same test cases against two agents:

1. ❌ **A weak agent with no tools** — it has no way to look up real customer data, so it
   guesses. The judge should give it **low scores**.
2. ✅ **The real agent with tools** — it looks up the actual data. The judge should give it
   **high scores**.

Seeing the weak agent fail is the point: it shows the rubric is actually doing its job, not
just stamping everything 1.0.

In [ ]:
from strands import Agent
from strands_evals import eval_task, Case, Experiment
from strands_evals.evaluators import OutputEvaluator
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise. Use the available tools to look up customer
information and process requests.

Important: Always verify the customer first, then check orders if needed."""


# Two agents under test: a weak one (no tools) and the real one (with tools).
@eval_task()
def weak_agent():
    """No tools — it can't look up real data, so it guesses. Expect LOW scores."""
    return Agent(
        system_prompt="You are a customer service agent. Answer as best you can.",
        callback_handler=None,
    )


@eval_task()
def good_agent():
    """Has tools — it looks up the real data. Expect HIGH scores."""
    return Agent(
        tools=[lookup_customer, get_order_history, process_refund],
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,
    )


# Test cases — the same questions for both agents
output_cases = [
    Case[str, str](
        name="order-status-check",
        input="I'm customer C-1001. Where is my USB-C Hub order?",
        expected_output="The USB-C Hub is shipped with tracking TRK-887766, estimated delivery 2025-05-06.",
    ),
    Case[str, str](
        name="delayed-order-empathy",
        input="I'm customer C-1002. My keyboard order is delayed and I'm frustrated!",
        expected_output="Acknowledge frustration, provide order status for the delayed mechanical keyboard with tracking info.",
    ),
    Case[str, str](
        name="unknown-customer",
        input="I'm customer C-9999. What are my orders?",
        expected_output="Inform the customer that no account was found with that ID and ask them to verify.",
    ),
]

# Define the evaluator rubric
output_evaluator = OutputEvaluator(
    rubric="""
    Evaluate the customer service response against the expected output:
    1. Accuracy — Does it contain the correct order/tracking details? Invented or
       missing details (e.g. a made-up tracking number) must score low.
    2. Tone — Is it professional and empathetic?
    3. Completeness — Does it fully address the customer's concern?

    Score 1.0 only if accuracy is correct AND tone and completeness are met.
    Score 0.5 if partially met (e.g. right tone but wrong/missing facts).
    Score 0.0 if the information is inadequate, invented, or incorrect.
    """,
    include_inputs=True,
)

print("❌ Weak agent (no tools) — expect LOW scores:")
weak_reports = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(weak_agent)
# Use .display() (static render) in notebooks — .run_display() opens an
# interactive Rich view that blocks the cell waiting for input.
weak_reports[0].display(include_actual_output=True)

print("\n✅ Real agent (with tools) — expect HIGH scores:")
good_reports = Experiment[str, str](cases=output_cases, evaluators=[output_evaluator]).run_evaluations(good_agent)
good_reports[0].display(include_actual_output=True)

print(
    f"\n📊 No-tools agent: {weak_reports[0].overall_score:.2f}"
    f"  vs  with-tools agent: {good_reports[0].overall_score:.2f}"
)
print("The gap is the eval doing its job — it catches the agent that guesses.")

---

## Part 2: Trajectory Evaluation — Did It Follow the Workflow?

The `TrajectoryEvaluator` checks that the agent called tools in the **correct order**. For a
refund, the policy is: look up the customer → check their order history → only then process
the refund.

Again we compare two agents to prove the eval works:

1. ❌ **A naive agent** told to "process refunds immediately" — it skips the verification
   steps. The trajectory eval should **fail** it.
2. ✅ **A steered agent** with an explicit ordered workflow — it follows the policy. The
   trajectory eval should **pass** it.

This is how you catch a missing guardrail before it reaches production.

In [ ]:
from strands_evals.evaluators import TrajectoryEvaluator
from strands_evals.extractors import tools_use_extractor
from strands_evals.types import TaskOutput

# Two system prompts: one that skips verification, one that enforces the workflow.
NAIVE_PROMPT = """You are a fast customer service agent. When a customer asks for a refund,
process it right away with process_refund. Don't waste time on extra lookups."""

STEERED_PROMPT = """You are a customer service agent. When processing refunds, you MUST:
1. First look up the customer
2. Then check their order history
3. Only then process the refund
Always follow this exact order."""


def run_with(prompt):
    """Build a task function that runs an agent with the given system prompt
    and captures the tools it called."""

    def task(case: Case) -> TaskOutput:
        agent = Agent(
            tools=[lookup_customer, get_order_history, process_refund],
            system_prompt=prompt,
            callback_handler=None,
        )
        response = agent(case.input)
        trajectory = tools_use_extractor.extract_agent_tools_used_from_messages(agent.messages)
        return TaskOutput(output=str(response), trajectory=trajectory)

    return task


# Cases with expected tool sequences
trajectory_cases = [
    Case[str, str](
        name="refund-workflow",
        input="Customer C-1001 wants a refund for order ORD-5521 ($79.99). Process it now.",
        expected_trajectory=["lookup_customer", "get_order_history", "process_refund"],
    ),
    Case[str, str](
        name="info-lookup-only",
        input="Look up customer C-1001 and tell me their order history.",
        expected_trajectory=["lookup_customer", "get_order_history"],
    ),
    Case[str, str](
        name="customer-lookup-only",
        input="Look up customer C-1002's account info.",
        expected_trajectory=["lookup_customer"],
    ),
]

# Create trajectory evaluator
trajectory_evaluator = TrajectoryEvaluator(
    rubric="""
    Evaluate whether the agent followed the expected tool sequence:
    - The expected tools should appear in order (extra tools in between are OK).
    - Score 1.0 if the expected sequence is followed correctly.
    - Score 0.5 if tools are called but in wrong order.
    - Score 0.0 if expected tools are missing entirely.
    """,
    include_inputs=True,
)

# Give evaluator context about available tools
sample_agent = Agent(tools=[lookup_customer, get_order_history, process_refund])
tool_descriptions = tools_use_extractor.extract_tools_description(sample_agent, is_short=True)
trajectory_evaluator.update_trajectory_description(tool_descriptions)

print("❌ Naive agent (skips verification) — expect a FAILED refund trajectory:")
naive_reports = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(NAIVE_PROMPT))
# .display() renders statically; .run_display() would block the cell (see Part 1).
naive_reports[0].display(include_actual_trajectory=True, include_expected_trajectory=True)

print("\n✅ Steered agent (enforced workflow) — expect all trajectories to PASS:")
steered_reports = Experiment[str, str](cases=trajectory_cases, evaluators=[trajectory_evaluator]).run_evaluations(run_with(STEERED_PROMPT))
steered_reports[0].display(include_actual_trajectory=True, include_expected_trajectory=True)

print(
    f"\n📊 Naive agent: {naive_reports[0].overall_score:.2f}"
    f"  vs  steered agent: {steered_reports[0].overall_score:.2f}"
)
print("The steering handlers are what close that gap — and the eval is what proves it.")

---

## 🎯 Try It Yourself

You've seen the eval catch a weak agent and a missing guardrail. Now extend it:

- Add a case where **no customer ID is given** — the agent should ask, not guess, so the
  expected trajectory is empty `[]`.
- Or tweak the `NAIVE_PROMPT` above and watch the scores move. That feedback loop — change
  the agent, rerun the eval, compare the numbers — is exactly how you harden an agent.

In [ ]:
# Challenge: Add a case where no customer ID is given
# Expected trajectory should be empty [] — the agent should ask, not guess

# new_case = Case[str, str](
#     name="missing-customer-id",
#     input="I want to return something I bought last week.",
#     expected_trajectory=[],
# )

# Your code here...

---

## What's Next

You've validated the agent works correctly. In **Module 7: Deploy**, you'll package it as a production service using Bedrock AgentCore with persistent memory.